# EMG data classification initial testing 

test space for extracting features

## Libraries 

In [1]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 
from spectrogram import plot_spectrogram 
from contraction_detection import detect_contractions
from scipy.optimize import brentq

matplotlib.use('QtAgg') 
mne.set_log_level("CRITICAL")

## Defining Initial Variables  

In [2]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']

inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/full_EEG_dataset" # REPLACE WITH OWN PATH 
current_index=0
inter_trigger_length=10
window = 50  # WINDOW FOR FEATURE EXTRACTION
step = 1 

Pre-Processing Steps

In [3]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,block):
    global raw_path
    global frq

    subject_name = subject + block 
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]


 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   


    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]

    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return epochs, df_triggers 

## Double Threshold

In [ ]:
def auxiliary_sequence(x, window):
    # generate auxillary sequence (basicall could replace with any feature)
    x = np.asarray(x, dtype=float)
    sq = x ** 2
    cum = np.concatenate([[0.0], np.cumsum(sq)])
    return cum[window:] - cum[:-window]


#-------------statistical methods for determining threshold based on data -------------
def solve_p_zeta(Pfa, m, r0):
    """Eq. (11), solved for P_zeta given Pfa, window m, count r0."""
    def f(p):
        return stats.binom.sf(r0 - 1, m, p) - Pfa
    return brentq(f, 1e-12, 1 - 1e-12)


def analytic_threshold(sigma_n, Pfa, m, r0, window):
    p_zeta = solve_p_zeta(Pfa, m, r0)
    zeta = sigma_n ** 2 * stats.chi2.ppf(1 - p_zeta, df=window)
    return zeta, p_zeta


def empirical_threshold(z_rest, Pfa, m, r0):
    p_zeta = solve_p_zeta(Pfa, m, r0)
    print(p_zeta)
    zeta = np.quantile(z_rest, 1 - p_zeta)
    return zeta, p_zeta


def detect_activation(z, zeta, m, r0, min_pulse_width):
    z = np.asarray(z)
    n = len(z)
    print("zeta:",zeta)

    above = z/100 > (zeta)
    cum = np.concatenate([[0], np.cumsum(above)])
    window_counts = cum[m:] - cum[:-m]

    # if counts of threshold pass r0 then window is considered active 
    window_is_active = window_counts >= r0

 
    active = np.zeros(n, dtype=bool)
    for i in np.flatnonzero(window_is_active):
        active[i:i + m] = True

    # reject short pulses based on window 
    return reject_short_pulses(above, min_pulse_width),z
#reject_short_pulses(above, min_pulse_width)


# function to reject short pulses 
def reject_short_pulses(active, min_width):
    active = active.copy()
    padded = np.r_[0, active.astype(int), 0]
    edges = np.flatnonzero(np.diff(padded))
    starts, ends = edges[0::2], edges[1::2]
    for s, e in zip(starts, ends):
        if e - s < min_width:
            active[s:e] = False
    return active


# go fromo boolean to actual time point of contractiosn 
def extract_contractions(active, fs):
    padded = np.r_[0, active.astype(int), 0]
    edges = np.flatnonzero(np.diff(padded))
    starts, ends = edges[0::2], edges[1::2]
    return [
        {"onset_s": s / fs, "offset_s": e / fs, "duration_s": (e - s) / fs}
        for s, e in zip(starts, ends)
    ]

def detect_contractions(emg, fs, rest_slice, Pfa=0.05, window=10, m=10,
                         r0=1, min_pulse_width=30):

    emg = np.asarray(emg, dtype=float)
    z = auxiliary_sequence(emg, window)

    rest_z = auxiliary_sequence(emg[rest_slice], window)
    zeta, p_zeta = empirical_threshold(rest_z, Pfa, m, r0)
   
    active_z,z_fin = detect_activation(z, zeta, m, r0, min_pulse_width)
    active = active_z
    events = extract_contractions(active, fs)

    return {"z": z_fin, "zeta": zeta, "p_zeta": p_zeta, "active": active_z,
            "events": events}



## Wavelet Transform

In [16]:
import pywt 

fs = 250   
wavelet = 'cmor1.5-1.0'

freqs_target = np.geomspace(5, 200, 64)   # example band

# convert frequencies to scales
scales = pywt.central_frequency(wavelet) * fs / freqs_target

coeffs, freqs = pywt.cwt(epoch_zygo, scales, wavelet, sampling_period=1/fs)


t = np.linspace(0, 1, 2251)


### plot wavelet transform 

In [17]:
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(12, 8),
    sharex=True,
    gridspec_kw={'height_ratios': [1, 2]}
)

# Signal
ax1.plot(t, epoch_zygo)
ax1.set_ylabel("Amplitude")
ax1.set_title("Signal")

# CWT magnitude
im = ax2.imshow(
    np.abs(coeffs),
    aspect='auto',
    origin='lower',
    extent=[time[0], time[-1], scales[0], scales[-1]]
)

ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Scale")
ax2.set_title("CWT Magnitude")

fig.colorbar(im, ax=ax2, label='|CWT|')

plt.tight_layout()
plt.show()

In [14]:
phase = np.angle(coeffs)

plt.figure(figsize=(12,6))

plt.pcolormesh(
    t,
    scales,
    phase,
    shading='auto'
)

plt.colorbar(label='Phase (rad)')
plt.xlabel('Time (s)')
plt.ylabel('Scale')
plt.title('CWT Phase')
plt.show()

NameError: name 't' is not defined

### wvt testing 

In [19]:
# CWT function 
import pywt
import numpy as np
import matplotlib.pyplot as plt

''' 
def detect_phase_crossings():
    return phase_lines

def detect_piecewise_constant_regions():
    return zones
'''

fs = 250   
wavelet = 'cmor1.5-1.0'

freqs_target = np.geomspace(5, 200, 64)   # example band

# convert frequencies to scales
scales = pywt.central_frequency(wavelet) * fs / freqs_target

coeffs, freqs = pywt.cwt(epoch_zygo, scales, wavelet, sampling_period=1/fs)

#--------MODULE 1--------
for scale, coeff_row in zip(scales, coeffs):
    phase = np.angle(coeff_row)
    

    phase_unwrapped = np.unwrap(phase)
    #phase_unwrapped % (2*np.pi) == 0
    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=(12, 6),
        sharex=True
    )

    # Unwrapped phase
    ax1.plot(t, phase_unwrapped)
    ax1.set_ylabel("Unwrapped phase")
    ax1.set_title(f"Phase at scale={scale:.2f}")

    # Wrapped phase
    ax2.plot(t, phase)
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Phase (rad)")

    plt.tight_layout()
    plt.show()
    
    # locate phase wraps / phase contours
    #phase_lines = detect_phase_crossings(phase) # the distance has a constant value

    # distance between consecutive crossings
    #d = np.diff(phase_lines)

    # segment regions where d is approximately constant
    #zones = detect_piecewise_constant_regions(d)


# module 1 
'''
The first module scans
scale by scale the WT phase to calculate the distance function
and then to detect a list of regularity zones, thus feeding the sec-
ond module with the relevant initial and final time instants for
each zone.

distance: he number of time samples between two consecutive phase lines
''' 

# build distance plot 
''' 
The time pairs bounding each regularity zone have to
be combined in order to build a distance plot covering all the time
axis of the signal.
'''

# module 2 
''' 
The second module, following a research tree,
goes through all the provided time pairs and builds all the admis-
sible solutions by juxtaposing contiguous time intervals. A decision
step, by integrating the WT module information, selects a final se-
quence of intervals, where the interval bounds are the searched
discontinuities.
'''

KeyboardInterrupt: 

## Classification Functions 

In [ ]:
# testing threshold based classification with windowing 
# define baseline threshold off of first 50 samples
# in a sliding window compare the signal (variance) to the baseline threshold 
# see classification 
def sliding_window(signal,window_size=50,):
    sig_len = np.shape(signal)
    baseline = np.mean(signal[0:window_size]**2)
    labels = []

    idx1 = 0 
    idx2 = window_size
    while (idx2 < sig_len):
        segment = signal[idx1:idx2]
        label = single_threshold(baseline,segment)
        labels.append(label)

    return labels 

## Feature Extraction 

### Feature extraction function 

In [86]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch):
     global frq
     global window 
     global step 

     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]


     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  

     # frequency features (did not use)
     X = np.fft.rfft(epoch_sw, axis=1)
     PSD = (1/(frq*window)) * np.abs(X)**2
     cumulative = np.cumsum(PSD, axis=1)
     total_power = cumulative[:, -1]
     half_power = total_power / 2

     indices = (cumulative >= half_power[:, None]).argmax(axis=1)
     frequencies = np.fft.rfftfreq(window, 1/frq)

     fmd = frequencies[indices]

  
     return var, rms, wl, fmd


#### Plotting function 

In [87]:
def plot_epoch(corr,zygo,Var_corr,Wl_corr,RMS_corr,
               Var_zygo,Wl_zygo,RMS_zygo):
    fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)

    ax_corr = ax[0].twinx()
    ax_zygo = ax[1].twinx()

    title = fig.suptitle(
        "Facial EMG Response Following Stimulus",
        fontsize=14,
    )

    corrugator_color = "#734F62"
    zygomatic_color = "#6F9359"

    time = np.linspace(0,10,2251)
    trigger_time = time[0]

                    
    # Plot Corr
    #ax[0].axvline(trigger_time, label="Trigger Channel", color="black")
    ax[0].plot(time,corr,  color=corrugator_color)
    ax[0].set_ylim(-200, 200)
    ax[0].set_ylabel("Corr",size=12)
    # ax[0].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))

    # Plot Zygo
    ax[1].plot(time, zygo, lw=1.5, color=zygomatic_color)
    ax[1].set_ylim(-200, 200)
    ax[1].set_ylabel("Zygo",size=12)
    # ax[1].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))


    # plotting features 
    ax_corr.plot(time,Var_corr,  color="magenta", alpha=0.7,)
    ax_corr.plot(time,-Wl_corr,  color="blue", alpha=0.7,)
    ax_corr.plot(time,RMS_corr*10,  color="black", alpha=0.7,) # multiply by 10 for scaling 
    ax_corr.set_ylim(-1500, 1500)

    ax_zygo.plot(time,Var_zygo,  color="magenta", label="variance", alpha=0.7,)
    ax_zygo.plot(time,-Wl_zygo,  color="blue", alpha=0.7)
    ax_zygo.plot(time,RMS_zygo*10,  color="black", alpha=0.7)
    ax_zygo.set_ylim(-1500, 1500)

    ax_zygo.axis("off")
    ax_corr.axis("off")

    plt.legend() 
    for a in ax:
        a.spines['top'].set_visible(False)
        a.spines['right'].set_visible(False)

        plt.tight_layout()
        #plt.show()
    return fig 

In [143]:
def plot_spec(epoch_corr,epoch_zygo):
    fig, axs = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
    fs=250

    title = fig.suptitle(
        "Facial EMG Response Following Stimulus",
        fontsize=14,
    )

    ax_corr = axs[0].twinx()
    ax_zygo = axs[1].twinx()

    corrugator_color = "#734F62"
    zygomatic_color = "#6F9359"

    time = np.linspace(0,10,2251)
    trigger_time = time[0]


    Pxx_c, freqs_c, bins_c, im_c = axs[0].specgram(epoch_corr, Fs=fs, NFFT=256, noverlap=128)
    axs[0].set_title("Corr")
    axs[0].set_ylabel("Frequency (Hz)")
    ax_corr.plot(time,epoch_corr,color=corrugator_color)
  #  ax_corr.plot(bins_c,freqs_c,color=corrugator_color)

    print(np.shape(freqs_c))

    Pxx_z, freqs_z, bins_z, im_z = axs[1].specgram(epoch_zygo, Fs=fs, NFFT=256, noverlap=128)
    axs[1].set_title("Zygo2")
    axs[1].set_ylabel("Frequency (Hz)")
    axs[1].set_xlabel("Time (s)")
    ax_zygo.plot(time,epoch_zygo,color=zygomatic_color)
    plt.tight_layout()
    plt.legend() 
    return fig

In [215]:
result

{'z': array([ 3.00203311,  2.20591082,  2.05141387, ..., 11.08900173,
        11.0800916 ,  7.10727788]),
 'zeta': 24.30796703817856,
 'p_zeta': 0.010206218313042853,
 'active': array([False, False, False, ..., False, False, False]),
 'events': [{'onset_s': 1.32, 'offset_s': 2.172, 'duration_s': 0.852},
  {'onset_s': 2.18, 'offset_s': 2.848, 'duration_s': 0.668},
  {'onset_s': 2.876, 'offset_s': 3.572, 'duration_s': 0.696}]}

In [305]:
def plot_double_threshold(epoch_corr,epoch_zygo):
    fig, axs = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
    fs=250

    title = fig.suptitle(
        "Facial EMG Response Following Stimulus",
        fontsize=14,
    )

    ax_corr = axs[0].twinx()
    ax_zygo = axs[1].twinx()

    corrugator_color = "#734F62"
    zygomatic_color = "#6F9359"

    time = np.linspace(0,10,2251)
    trigger_time = time[0]
    
    print("Zygo")
    result_zygo = detect_contractions(epoch_zygo, 250, slice(0, 10), Pfa=0.05, window=10, m=10,
                         r0=2, min_pulse_width=20, threshold_mode="empirical")

    print("Corr")
    result_corr = detect_contractions(epoch_corr, 250, slice(0, 10), Pfa=0.05, window=10, m=10,
                         r0=2, min_pulse_width=20, threshold_mode="empirical")
   
    axs[0].plot(epoch_corr,color=corrugator_color,label="corr")
    axs[0].plot(result_corr['z']/100,label="signal power")
    axs[0].set_title("Corr")
    axs[0].set_ylim(-200,200)
    axs[0].plot(result_corr['active']*100,label="output labels")
    axs[0].axhline(result_corr['zeta'],linestyle='--',color="red",lw=.8,label="corr threshold")
    #ax_corr.set_ylim(-2,2)

  #  ax_corr.plot(bins_c,freqs_c,color=corrugator_color)

    axs[1].plot(epoch_zygo,color=zygomatic_color)
    axs[1].set_title("Zygo")
    axs[1].plot(result_zygo['z']/100,label="signal power")
    axs[1].set_ylim(-200,200)
    axs[1].set_xlabel("Time (s)")
    axs[1].plot(result_zygo['active']*100,label="output labels")
    axs[1].axhline(result_zygo['zeta'],linestyle='--',color="red",lw=.8,label="threshold")
   # ax_zygo.set_ylim(-2,2)
    plt.tight_layout()
    axs[1].legend() 
    return fig

In [100]:
def on_key(event):
    global current_index, fig, subject_epoch, subject, block

    if event.key == 'right':
        current_index = (current_index + 1) % len(subject_epoch)
    elif event.key == 'left':
        current_index = (current_index - 1) % len(subject_epoch)
    elif event.key == 'escape':
        print("Quitting the plot!")
        plt.close(fig)
        return

    epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
    epoch_corr = np.squeeze(subject_epoch[current_index].get_data(picks=['Corr']))

    epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
    epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)

    plt.close(fig)
    #fig = plot_epoch(epoch_corr, epoch_zygo, epoch_var_corr, epoch_wl_corr, epoch_rms_corr,
                   #  epoch_var_zygo, epoch_wl_zygo, epoch_rms_zygo)
    #fig = plot_spec(epoch_corr,epoch_zygo)
    fig = plot_double_threshold(epoch_corr,epoch_zygo)
    

    
    fig.canvas.mpl_connect('key_press_event', on_key)   
    fig.suptitle(f"subject: {subject} | block: {block} | epoch: {current_index}", fontsize=14)
    plt.show(block=False)
    print(current_index)


### Loop through all files 

In [ ]:
#define data frame if want to store all features 
def define_df():
    features = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Nap Number",
            "Triggers_Order_Nap", # epochs 
            "True_Muscle_Activated",
            "Num_Contractions_Zygo",
            "Num_Contractions_Corr",
            "WL_Zygo", # three features being used 
            "Var_Zygo",
            "RMS_Zygo", 
            "WL_Corr",
            "Var_Corr",
            "RMS_Corr", 
        ]
        ) 
    return features 

In [ ]:
# extracting features for classification 
i = 0 
# features_results_mat = [] if want to fill data frame with features for each subject 

for root,dirs,files in os.walk(raw_path): # loop through file 
    for file in files:
        if "dpa" not in file and ".DS_Store" not in file: # so only take each subject once 
            subject = file.split(".")[0][:-2]
            block = file.split(".")[0][-2:]
            print(file)
            print(subject+block)


            # need to leave out these subjects 
            if (subject == "NL02IF" or 
                subject == "NL05WW" or 
                subject == "NL01SS" or 
                subject == "RL11JH" or 
                subject == "RL12JL" or 
                subject == "RL07BR"):
                continue
            
            # features = def_df() # define dataframe 
            subject_epoch, _ = pre_process_subjets(subject,block) # preprocess subject to get epochs in a block 
            i += 1
            for t in range(len(subject_epoch)): 
                # extract epoch  
                epoch_zygo = np.squeeze(subject_epoch[t].get_data(picks=['Zygo']))
                epoch_corr = np.squeeze(subject_epoch[t].get_data(picks=['Corr']))
                
                # get features for epoch 
                epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
                epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)


                

                # fill data frame 
                ''' 
                features.loc[t] = [
                    subject, 
                    int(block),
                    t + 1,
                    subject_epoch[t].metadata['True_activation'].iloc[0],
                    subject_epoch.metadata.iloc[t]["Nb_Zygo"],
                    subject_epoch.metadata.iloc[t]["Nb_Corr"],
                    epoch_wl_zygo,
                    epoch_var_zygo, 
                    epoch_rms_zygo, 
                    epoch_wl_corr,
                    epoch_var_corr, 
                    epoch_rms_corr, 
     
                ]
                features_results_mat.append(features)
                '''


print(f"{i} Subjects & Naps Processed")

### Plot a single subject and epoch

In [42]:
# define subject 
subject = 'RL09PC'
block = '01' #nap number
epoch = 3 
subject_epoch, _ = pre_process_subjets(subject,block)

In [192]:
# get features 
epoch_zygo = np.squeeze(subject_epoch[epoch].get_data(picks=['Zygo']))
epoch_corr = np.squeeze(subject_epoch[epoch].get_data(picks=['Corr']))

# get features for epoch 
epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)

In [11]:
fig = plot_epoch(epoch_corr,epoch_zygo,epoch_var_corr,epoch_wl_corr,epoch_rms_corr,
           epoch_var_zygo,epoch_wl_zygo,epoch_rms_zygo)

title = fig.suptitle(
        f"Facial EMG Response Following Stimulus | subject: {subject} | block: {block} | epoch: {epoch}",
        fontsize=14,
    )
plt.show()

### Loop through all epochs of a single subject

In [6]:
subject = 'RL09PC'
block = '01' #nap number
subject_epoch, _ = pre_process_subjets(subject,block) # preprocess subject to get epochs in a block 
current_index = 0 

In [8]:
# extract epoch  
epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
epoch_corr = np.squeeze(subject_epoch[current_index].get_data(picks=['Corr']))

# get features for epoch 
epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)

fig = plot_epoch(epoch_corr,epoch_zygo,epoch_var_corr,epoch_wl_corr,epoch_rms_corr,
        epoch_var_zygo,epoch_wl_zygo,epoch_rms_zygo)

fig.canvas.mpl_connect('key_press_event', on_key)


title = fig.suptitle(
    f"subject: {subject} | block: {block} | epoch: {current_index}",
    fontsize=14,
)
plt.show()

NameError: name 'get_features' is not defined

### Plot spectrogram

In [193]:
epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
epoch_corr = np.squeeze(subject_epoch[current_index].get_data(picks=['Corr']))

fig = plot_spec(epoch_corr,epoch_zygo)

fig.canvas.mpl_connect('key_press_event', on_key)


plt.show()


(129,)


/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_14007/3349487579.py:34: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


### Plot Double Threshold Test 

In [306]:
epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
epoch_corr = np.squeeze(subject_epoch[current_index].get_data(picks=['Corr']))

fig = plot_double_threshold(epoch_corr,epoch_zygo)

fig.canvas.mpl_connect('key_press_event', on_key)


plt.show()

Zygo
0.03677143788746508
zeta: 21.384828721680115
Corr
0.03677143788746508
zeta: 28.230830703453226


KeyboardInterrupt: 